# 03 Build Store Dimension Table (Silver Layer)

This notebook reads physical store metadata from the Bronze layer (`stores.parquet`), standardizes the keys, casts datatypes, and ensures geographical names are kept in their raw proper formats.

In [1]:
import os
from pathlib import Path
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# Locate lakehouse root dynamically
lakehouse_root = None
for p in [Path.cwd()] + list(Path.cwd().parents):
    if (p / "data_lake").is_dir() and (p / "notebooks").is_dir():
        lakehouse_root = p
        break

if lakehouse_root is None:
    for search_path in [
        "/content/Demand_forecasting/inventory-lakehouse",
        "/content/inventory-lakehouse",
        "/content/drive/MyDrive/Demand_forecasting/inventory-lakehouse"
    ]:
        if Path(search_path).is_dir():
            lakehouse_root = Path(search_path)
            break

if lakehouse_root is None:
    lakehouse_root = Path("D:/Demand_forecasting/inventory-lakehouse")

project_root = lakehouse_root.parent

In [2]:
BRONZE_DIR = project_root / "inventory-lakehouse" / "data_lake" / "bronze"
SILVER_DIR = project_root / "inventory-lakehouse" / "data_lake" / "silver"
store_out_dir = SILVER_DIR / "store_dimension"
store_out_dir.mkdir(parents=True, exist_ok=True)

print("Starting store_dimension pipeline...")
df_stores = pq.read_table(BRONZE_DIR / "stores.parquet").to_pandas()
print(f"Loaded {len(df_stores):,} stores records from Bronze.")

df_stores = df_stores.rename(columns={
    "store_nbr": "store_id",
    "type": "store_type"
})
df_stores["store_id"] = df_stores["store_id"].astype("int32")
df_stores["cluster"] = df_stores["cluster"].astype("int32")

df_stores_dim = df_stores[["store_id", "city", "state", "store_type", "cluster"]]

table_stores = pa.Table.from_pandas(df_stores_dim, preserve_index=False)
pq.write_table(table_stores, store_out_dir / "store_dimension.parquet", compression='snappy')
print(f"Completed store_dimension pipeline.")
print(f"Store Dimension Rows: {len(df_stores_dim):,}")

Starting store_dimension pipeline...
Loaded 54 stores records from Bronze.
Renamed columns: store_nbr -> store_id, type -> store_type.
Preserved Spanish city/state names as proper nouns.
Completed store_dimension pipeline in 0.03s.
Store Dimension Rows: 54
